# File xây dựng vector store, lưu trữ các schema table

## Pipeline bao gồm
- Xây dựng collection
- Thêm data và metadata vào collection
- Lựa chọn kĩ thuật RAG

In [ ]:
import re
import json
import torch
import pandas as pd
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer

d:\python-workspace\NL2SQL-chat\myenv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Tạo và thêm dữ liệu vào vector store

In [2]:
class EmbeddingModel:
    def __init__(self, model, device):
        self.model = SentenceTransformer(model, device=device)

    def embed_documents(self,data):
        return self.model.encode_document(data)

    def embed_query(self, query):
        return self.model.encode_query(query)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
embed_model = EmbeddingModel("all-MiniLM-L6-v2", device)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10649.64it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
vector_store = Chroma(
    collection_name="first_collection",
    persist_directory="../vector-store",
    embedding_function=embed_model,
    collection_metadata={"hnsw:space": "cosine"}
)

In [ ]:
list_schema = ""
list_metadata = ""
with open("../schema_data/table_schema.txt", mode='r', encoding='utf-8') as f:
    list_schema = f.read()
with open("../schema_data/better_metadata.txt", mode='r', encoding='utf-8') as f:
    list_metadata = f.read()
list_schema = re.findall(r"CREATE TABLE.*?\);", list_schema, flags=re.DOTALL)
list_metadata = re.findall(r"{.*?}", list_metadata, flags=re.DOTALL)
print(f"Count schema: {len(list_schema)}")
print(f"Count metadata: {len(list_metadata)}")

In [ ]:
metadatas = [{"raw_context": schema} for schema in list_schema]
vector_store.add_texts(
    texts=list_metadata,
    metadatas=metadatas
)
print(f"Adding {len(list_schema)} table schema successful")

### Kiểm tra RAG

In [ ]:
query = "Find the total revenue generated from orders that were paid using 'Credit Card' and have already been shipped"

# Tìm kiếm top 2 bảng liên quan nhất
results = vector_store.similarity_search_with_relevance_scores(query, k=10)
print(results)
for i, doc in enumerate(results):
    print(f"\nKết quả {i+1}:")
    print(f"Điểm số liên quan: {doc[1]}")
    print(f"Nội dung tìm thấy: {doc[0].page_content}") # Đây là phần metadata enrichment
    # print(f"SQL Schema để ném vào LLM: {doc[0].metadata['raw_context']}")

[Document(id='c1651472-8ce3-4920-aa55-cee9ce6068ba', metadata={'raw_context': "CREATE TABLE Orders (\n    order_id SERIAL PRIMARY KEY,\n    user_id INT,\n    order_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,\n    total_amount DECIMAL(12, 2) NOT NULL,\n    status VARCHAR(50) DEFAULT 'Pending', \n    FOREIGN KEY (user_id) REFERENCES Users(user_id) ON DELETE SET NULL\n);"}, page_content='{\n"table_name": "Orders",\n"table_description": "Stores information about customer transactions, including the total monetary value, current fulfillment status, and the timestamp of when the purchase was executed.",\n"columns_summary": "order_id: unique auto-incrementing identifier for the order | user_id: foreign key linking the order to the purchasing user | order_date: timestamp recording when the order was placed | total_amount: total calculated cost of the purchase | status: current tracking state of the order defaulting to Pending"\n}'), Document(id='a006f772-092f-4ae0-9667-6b7cfa0b3a02', metadata={'

TypeError: 'Document' object is not subscriptable

In [7]:
from sentence_transformers import CrossEncoder

# ==========================================
# KHỞI TẠO MÔ HÌNH RERANKER
# ==========================================
print("Đang tải mô hình Cross-Encoder...")
reranker_model = CrossEncoder('BAAI/bge-reranker-base', device='cuda')

def two_stage_retrieval(query, vector_store, stage1_k=8, stage2_k=5):
    """
    Hàm truy xuất 2 giai đoạn:
    - Giai đoạn 1: Dùng ChromaDB lấy top 10 (chấp nhận điểm thấp, miễn là không sót).
    - Giai đoạn 2: Dùng Cross-Encoder chấm điểm lại và lấy top 5 đưa vào LLM.
    """
    
    # ---------------------------------------------------------
    # GIAI ĐOẠN 1: VECTOR RETRIEVAL (LẤY DIỆN RỘNG)
    # ---------------------------------------------------------
    # Gọi hàm similarity_search bình thường, không cần quan tâm ngưỡng điểm.
    # Lấy hẳn 15-20 bảng để đảm bảo bảng đúng (dù điểm 0.2) vẫn lọt vào danh sách này.
    print(f"\n[STAGE 1] Đang quét VectorDB lấy Top {stage1_k}...")
    stage1_docs = vector_store.similarity_search(query, k=stage1_k)
    
    # ---------------------------------------------------------
    # GIAI ĐOẠN 2: CROSS-ENCODER RERANKING (CHẤM ĐIỂM TINH)
    # ---------------------------------------------------------
    print("[STAGE 2] Đang Rerank lại các tài liệu...")
    
    # Tạo danh sách các cặp [Câu hỏi, Nội dung tài liệu]
    # Cross-Encoder yêu cầu đầu vào phải là một cặp đi liền nhau để nó so sánh chéo.
    sentence_pairs = [[query, doc.page_content] for doc in stage1_docs]
    # print("This is sentencec pairs: \n", sentence_pairs)

    # Mô hình dự đoán điểm số mới cho toàn bộ cặp này cùng lúc
    rerank_scores = reranker_model.predict(sentence_pairs)
    # print("This is rerank scores: \n", rerank_scores)
    
    # Ghép điểm số mới vào tài liệu tương ứng
    # Lưu ý: Convert điểm số thành float để dễ in ấn và so sánh
    scored_docs = []
    for doc, score in zip(stage1_docs, rerank_scores):
        scored_docs.append((doc, float(score)))
    
    # Sắp xếp danh sách giảm dần theo điểm số mới (từ cao xuống thấp)
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    
    # Chỉ lấy Top K tài liệu xuất sắc nhất (ví dụ: 5 bảng)
    final_docs = scored_docs[:stage2_k]
    # print("This is final docs: \n", final_docs)
    
    return final_docs

# ==========================================
# CHẠY THỬ NGHIỆM
# ==========================================
user_query = "Find the total revenue generated from orders that were paid using 'Credit Card' and have already been shipped"

# Giả định vector_store là đối tượng Chroma của bạn
final_results = two_stage_retrieval(user_query, vector_store, stage1_k=8, stage2_k=5)
print(final_results)
print("\n[KẾT QUẢ CUỐI CÙNG SAU KHI RERANK]")
for i, (doc, score) in enumerate(final_results):
    content = doc.page_content
    print(f"Top {i+1} | Điểm Rerank: {score:.4f} | Nội dung bảng: \n", content)

Đang tải mô hình Cross-Encoder...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 7815.76it/s]
XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...
[(Document(id='edb042b2-daf9-4c5b-9355-c5e7f716d62d', metadata={'raw_context': "CREATE TABLE Payments (\n    payment_id SERIAL PRIMARY KEY,\n    order_id INT,\n    payment_method VARCHAR(50) NOT NULL, \n    amount DECIMAL(12, 2) NOT NULL,\n    status VARCHAR(50) DEFAULT 'Pending', \n    transaction_date TIMESTAMP DEFAULT CURRENT_TIMESTAMP,\n    FOREIGN KEY (order_id) REFERENCES Orders(order_id) ON DELETE CASCADE\n);"}, page_content='{\n"table_name": "Payments",\n"table_description": "Stores financial transaction records associated with customer orders, tracking the payment method used, monetary amount, and current processing status.",\n"columns_summary": "payment_id: unique auto-incrementing identifier for the transaction | order_id: foreign key linking to the associated customer order | payment_method: the specific mode of payment used such as credit card or digital wallet | amount: the total monetary

In [8]:
# vector_store.delete_collection()

### Đánh giá RAG truyền thống và RAG Re-ranker

In [9]:
eval_data = pd.read_csv("../schema_data/evaluate_rag.csv")
eval_data.head()

,english_query,RAG_tables
0,Show all user profiles created in 2024.,Users
1,List the names of all product categories.,Categories
2,Find products with a price lower than $50.,Products
3,Count the number of pending orders.,Orders
4,Retrieve all reviews with a rating of 5.,Reviews


In [10]:
def count_table(vector_store, query, true_answer, rag_type='original'):
    true_answer = true_answer.split()
    rag_answer_dict = []
    if rag_type=='original':
        rag_answer = vector_store.similarity_search_with_relevance_scores(query, k=5)
        for doc in rag_answer:
            rag_answer_dict.append(json.loads(doc[0].page_content))
    elif rag_type=='reranker':
        rag_answer = two_stage_retrieval(query, vector_store)
        for doc in rag_answer:
            rag_answer_dict.append(json.loads(doc[0].page_content))
    else:
        print("Sai loại RAG")
        return -1
    rag_answer_lst = [data.get("table_name") for data in rag_answer_dict]
    count = 0
    for table_name in rag_answer_lst:
        if table_name in true_answer:
            count += 1
    return count == len(true_answer)

In [11]:
eval_result_original = eval_data.copy()
eval_result_original['result'] = eval_result_original.apply(lambda row: count_table(vector_store, row['english_query'], row['RAG_tables']), axis=1)
print("Truy vấn schema đúng: ", len(eval_result_original[eval_result_original['result'] == True]))
eval_result_original

Truy vấn schema đúng:  32


,english_query,RAG_tables,result
0,Show all user profiles created in 2024.,Users,True
1,List the names of all product categories.,Categories,True
2,Find products with a price lower than $50.,Products,True
3,Count the number of pending orders.,Orders,True
4,Retrieve all reviews with a rating of 5.,Reviews,True
5,List products belonging to the 'Electronics' c...,Products Categories,True
6,Show the full name and email of user with ID 10.,Users,True
7,Get the total amount of all orders placed by u...,Users Orders,False
8,List all items currently in the shopping cart ...,Shopping_Cart Cart_Items Products,True
9,Find the average price of products in each cat...,Products Categories,False


In [12]:
eval_result_reranker = eval_data.copy()
eval_result_reranker['result'] = eval_result_reranker.apply(lambda row: count_table(vector_store, row['english_query'], row['RAG_tables'], rag_type='reranker'), axis=1)
print("Truy vấn schema đúng: ", len(eval_result_reranker[eval_result_reranker['result'] == True]))
eval_result_reranker


[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank lại các tài liệu...

[STAGE 1] Đang quét VectorDB lấy Top 8...
[STAGE 2] Đang Rerank 

,english_query,RAG_tables,result
0,Show all user profiles created in 2024.,Users,True
1,List the names of all product categories.,Categories,True
2,Find products with a price lower than $50.,Products,True
3,Count the number of pending orders.,Orders,True
4,Retrieve all reviews with a rating of 5.,Reviews,True
5,List products belonging to the 'Electronics' c...,Products Categories,True
6,Show the full name and email of user with ID 10.,Users,True
7,Get the total amount of all orders placed by u...,Users Orders,False
8,List all items currently in the shopping cart ...,Shopping_Cart Cart_Items Products,False
9,Find the average price of products in each cat...,Products Categories,True
